In [4]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for flasher scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_V13 = 1    # 8B
POINTS_V14 = 5.5833333   # 8B+
POINTS_V15 = 33.5   # 8C
POINTS_V16 = 1    # 8C+

TOP_LABELS = 12               # label top N names
JITTER = 0.025
LABELED_X_SPREAD = 1.15

HIGHLIGHT_NAME = "Adam Ondra" # set to None if you don't want special highlight
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/20/hard-boulder-problem-flashes"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

font_to_v = {"8B": 13, "8B+": 14, "8C": 15, "8C+": 16}
score_map = {13: POINTS_V13, 14: POINTS_V14, 15: POINTS_V15, 16: POINTS_V16}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade", "climb"}]
df = df[keep].copy()


df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["V"] = df["Grade_clean"].map(font_to_v)
df["Score"] = df["V"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions (x is meaningless) ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))
X_OFFSET = -0.074
rank["x"] = rank["x"] + X_OFFSET




# --- Choose which names to label (top N, plus highlight name if present) ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Improve label spacing ---
# Alternate labeled names left/right so nearby labels don't stack
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points farther horizontally for extra separation
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left-side vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Boulder Flashers (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.12, 0.12],
    ),
    yaxis=dict(
        title="Flash Score",
        type="log",
        tickfont=dict(size=20),
        showgrid=False   # removes horizontal gridlines
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)


fig.show()

out_html = "flashers_vertical_strip_outlier.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: flashers_vertical_strip_outlier.html


In [ ]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for flasher scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_V13 = 1    # 8B
POINTS_V14 = 5   # 8B+
POINTS_V15 = 18   # 8C
POINTS_V16 = 1    # 8C+

TOP_LABELS = 12               # label top N names
JITTER = 0.025
LABELED_X_SPREAD = 1.15

HIGHLIGHT_NAME = "Adam Ondra" # set to None if you don't want special highlight
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/20/hard-boulder-problem-flashes"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

font_to_v = {"8B": 13, "8B+": 14, "8C": 15, "8C+": 16}
score_map = {13: POINTS_V13, 14: POINTS_V14, 15: POINTS_V15, 16: POINTS_V16}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade", "climb"}]
df = df[keep].copy()



df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["V"] = df["Grade_clean"].map(font_to_v)
df["Score"] = df["V"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions (x is meaningless) ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))
X_OFFSET = -0.074
rank["x"] = rank["x"] + X_OFFSET




# --- Choose which names to label (top N, plus highlight name if present) ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Improve label spacing ---
# Alternate labeled names left/right so nearby labels don't stack
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points farther horizontally for extra separation
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left-side vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Boulder Flashers (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.12, 0.12],
    ),
    yaxis=dict(
        title="Flash Score",
        tickfont=dict(size=20),
        showgrid=False   # removes horizontal gridlines
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)


fig.show()

out_html = "flashers_vertical_strip_outlier.html"
fig.write_html(out_html)
print("Saved:", out_html)

KeyError: 'Column not found: Score'

In [3]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for ascent scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_V16 = 1    # 8C+
POINTS_V17 = 5    # 9A
POINTS_V18 = 18   # 9A+

TOP_LABELS = 12               # label top N names
JITTER = 0.025
LABELED_X_SPREAD = 1.15

HIGHLIGHT_NAME = "Elias Iagnemma"  # set to None if you don't want special highlight
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/14/hard-boulder-problem-ascents"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

font_to_v = {"8C+": 16, "9A": 17, "9A+": 18}
score_map = {16: POINTS_V16, 17: POINTS_V17, 18: POINTS_V18}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade", "climb"}]
df = df[keep].copy()


df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["V"] = df["Grade_clean"].map(font_to_v)
df["Score"] = df["V"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions (x is meaningless) ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))
X_OFFSET = -0.074
rank["x"] = rank["x"] + X_OFFSET


# --- Choose which names to label (top N, plus highlight name if present) ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Improve label spacing ---
# Alternate labeled names left/right so nearby labels don't stack
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points farther horizontally for extra separation
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left-side vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Hardest Boulder Ascensionists (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.12, 0.12],
    ),
    yaxis=dict(
        title="Ascent Score",
        tickfont=dict(size=20),
        showgrid=False
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "hardest_boulder_ascents_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: hardest_boulder_ascents_vertical_strip.html


In [18]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for ascent scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_9B = 1
POINTS_9B_PLUS = 1
POINTS_9C = 1

TOP_LABELS = 12               # label top N names
JITTER = 0.025
LABELED_X_SPREAD = 1.15

HIGHLIGHT_NAME = "Adam Ondra" # set to None if you don't want special highlight
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/5/hard-sport-climbing-ascents"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

score_map = {
    "9B": POINTS_9B,
    "9B+": POINTS_9B_PLUS,
    "9C": POINTS_9C,
}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade"}]
df = df[keep].copy()

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["Score"] = df["Grade_clean"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions (x is meaningless) ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))
X_OFFSET = -0.074
rank["x"] = rank["x"] + X_OFFSET


# --- Choose which names to label (top N, plus highlight name if present) ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# Always label Will Bosi too
rank.loc[rank["Climber"] == "Will Bosi", "label"] = "Will Bosi"

# --- Improve label spacing ---
# Alternate labeled names left/right so nearby labels don't stack
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points farther horizontally for extra separation
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD


# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left-side vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Hardest Sport Ascensionists (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.12, 0.12],
    ),
    yaxis=dict(
        title="Sport Score",
        tickfont=dict(size=20),
        showgrid=False
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "hardest_sport_ascents_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: hardest_sport_ascents_vertical_strip.html


In [17]:
# Self-contained: Vertical "strip" scatter (x doesn't mean anything) for onsight scores

import re
from io import StringIO

import numpy as np
import pandas as pd
import requests

import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "browser"

# ============================
# 🎛️ TUNE THESE PARAMETERS
# ============================
POINTS_8C_PLUS = 1
POINTS_9A = 1

TOP_LABELS = 12               # label top N names
JITTER = 0.025
LABELED_X_SPREAD = 1.15

HIGHLIGHT_NAME = "Adam Ondra" # set to None if you don't want special highlight
RANDOM_SEED = 7
# ============================

URL = "https://climbing-history.org/list/37/hard-sport-climbing-onsights"

def normalize_grade(g):
    if pd.isna(g):
        return None
    s = str(g)
    s = s.replace("\xa0", " ").replace("\u2009", " ").replace("\u202f", " ")
    s = re.sub(r"\s+", "", s).upper()
    s = s.replace("PLUS", "+")
    return s

score_map = {
    "8C+": POINTS_8C_PLUS,
    "9A": POINTS_9A,
}

# --- Load table ---
html = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"}).text
df = pd.read_html(StringIO(html), flavor="bs4")[0].copy()

df.columns = [str(c).strip() for c in df.columns]
keep = [c for c in df.columns if c.lower() in {"climber", "grade"}]
df = df[keep].copy()

df["Grade_clean"] = df["Grade"].apply(normalize_grade)
df["Score"] = df["Grade_clean"].map(score_map)
df = df.dropna(subset=["Climber", "Score"])

# total score per climber
rank = (
    df.groupby("Climber")["Score"]
      .sum()
      .sort_values(ascending=False)
      .reset_index()
      .rename(columns={"Score": "TotalScore"})
)

# --- Jitter x positions (x is meaningless) ---
rng = np.random.default_rng(RANDOM_SEED)
rank["x"] = rng.uniform(-JITTER, JITTER, size=len(rank))
X_OFFSET = -0.074
rank["x"] = rank["x"] + X_OFFSET


# --- Choose which names to label (top N, plus highlight name if present) ---
rank["label"] = ""
top_idx = rank.head(TOP_LABELS).index
rank.loc[top_idx, "label"] = rank.loc[top_idx, "Climber"]

if HIGHLIGHT_NAME and (rank["Climber"] == HIGHLIGHT_NAME).any():
    rank.loc[rank["Climber"] == HIGHLIGHT_NAME, "label"] = HIGHLIGHT_NAME

# --- Improve label spacing ---
# Alternate labeled names left/right so nearby labels don't stack
rank["textpos"] = "middle right"
label_mask = rank["label"] != ""
label_indices = rank.index[label_mask].tolist()

for i, idx in enumerate(label_indices):
    rank.at[idx, "textpos"] = "middle right" if i % 2 == 0 else "middle left"

# Spread labeled points farther horizontally for extra separation
rank.loc[label_mask, "x"] = rank.loc[label_mask, "x"] * LABELED_X_SPREAD

# ============================
# MANUAL LABEL FIXES
# ============================

mask_domen = rank["Climber"].str.contains("Domen", case=False, na=False)
rank.loc[mask_domen, "textpos"] = "top center"
rank.loc[mask_domen, "x"] += 0.01

mask_puig = rank["Climber"].str.contains("Puigblanque", case=False, na=False)
rank.loc[mask_puig, "textpos"] = "middle left"
rank.loc[mask_puig, "x"] -= 0.02

mask_rogora = rank["Climber"].str.contains("Rogora", case=False, na=False)
rank.loc[mask_rogora, "textpos"] = "top center"

mask_patxi = rank["Climber"].str.contains("Usobiaga", case=False, na=False)
rank.loc[mask_patxi, "textpos"] = "middle right"
rank.loc[mask_patxi, "x"] += 0.02

# ============================

# split highlight vs others
is_highlight = (
    rank["Climber"] == HIGHLIGHT_NAME
) if HIGHLIGHT_NAME else pd.Series(False, index=rank.index)

others = rank[~is_highlight].copy()
hilite = rank[is_highlight].copy()

# --- Plot ---
fig = go.Figure()

fig.add_layout_image(
    dict(
        source="C:/Users/joeka/Downloads/360_F_282740083_aA6llyff0Ck1LAH5D69x6OaKbxM2JxIj.jpg",
        xref="paper",
        yref="paper",
        x=0,
        y=1,
        sizex=1,
        sizey=1,
        sizing="stretch",
        opacity=1,
        layer="below"
    )
)

fig.add_trace(go.Scatter(
    x=others["x"],
    y=others["TotalScore"],
    mode="markers+text",
    text=others["label"],
    textposition=others["textpos"],
    hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
    customdata=np.stack([others["Climber"]], axis=-1),
    marker=dict(size=10, opacity=0.85),
    name="Climbers",
))

if len(hilite):
    fig.add_trace(go.Scatter(
        x=hilite["x"],
        y=hilite["TotalScore"],
        mode="markers+text",
        text=hilite["label"],
        textposition=hilite["textpos"],
        hovertemplate="<b>%{customdata[0]}</b><br>Score: %{y}<extra></extra>",
        customdata=np.stack([hilite["Climber"]], axis=-1),
        marker=dict(size=14, opacity=1.0, symbol="diamond"),
        name=HIGHLIGHT_NAME,
    ))

# Left-side vertical axis line
fig.add_shape(
    type="line",
    x0=0.02, x1=0.02,
    y0=0, y1=1,
    xref="paper",
    yref="paper",
    line=dict(width=3, color="white"),
)

fig.update_layout(
    title="Best Hardest Onsighters (1-D vertical strip plot)",
    template="plotly_dark",
    xaxis=dict(
        title=None,
        showticklabels=False,
        showgrid=False,
        zeroline=False,
        range=[-0.12, 0.12],
    ),
    yaxis=dict(
        title="Onsight Score",
        tickfont=dict(size=20),
        showgrid=False
    ),
    margin=dict(l=80, r=40, t=70, b=60),
    showlegend=True,
)

fig.show()

out_html = "hardest_sport_onsights_vertical_strip.html"
fig.write_html(out_html)
print("Saved:", out_html)

Saved: hardest_sport_onsights_vertical_strip.html
